# Access LSST Alert Data via BigQuery

The Pitt-Google Alert Broker's tabular alert and value-added alert data is stored in [BigQuery](https://docs.cloud.google.com/bigquery/docs/introduction). The dataset and tables names containing LSST data are outlined in our [data listings page](https://mwvgroup.github.io/pittgoogle-client/listings.html#id4).

## Table of Contents

- [Alert tables (overview)](#alert-tables)
- [Value-added tables (overview)](#value-added-tables)
- [SQL query examples](#sql-query-examples)

In [1]:
import pittgoogle

# define the Pitt-Google Alert Broker's GCP project name
pgb_project = "pitt-alert-broker"

<a id="alert-tables"></a>
## Alert tables

The alert tables served by the Pitt-Google Alert Broker represent an archive of the `lsst-alerts` Pub/Sub stream. These tables preserve the structure of the original alert packets (excluding cutouts), including all nested and repeated fields defined in the versioned LSST alert schema. To improve query performance and reduce query costs, these tables are partitioned by the field: `kafkaPublishTimestamp` and clustered by the fields: `healpix9`, `healpix19`, and `healpix29`. 

### Partitions

Partitioning an alert table by the `kafkaPublishTimestamp` field divides the table into daily segments, effectively reducing the number of bytes read by a query when a date (or date range) is specified.

### Clustering

The `healpix9`, `healpix19`, and `healpix29` fields represent the [HEALPix](https://healpix.sourceforge.io) spatial index of the diaSource at three commonly used resolutions ($N_{side} = 2^9$, $2^{19}$, and $2^{29}$). Clustering the alert tables by these fields sorts data within each partition by the HEALPix index, placing nearby sky regions physically close in storage. Depending on the search scale, queries can target coarse (order 9) or fine (order 29) spatial partitions.

The diagram below, obtained from Google's documentation on the [combination of clustered and partitioned tables](https://docs.cloud.google.com/bigquery/docs/clustered-tables#combine-clustered-partitioned-tables) demonstrates how more finely grained sorting is achieved using these methods.

![title](clustering-and-partitioning-tables.png)

The LSST alert schema v10_0 can be accessed [here](https://github.com/lsst/alert_packet/tree/main/python/lsst/alert/packet/schema/10/0) for your convenience. You can alternatively explore the schema of the `alerts_v10_0` table in the cell below:

In [2]:
alerts_tbl = pittgoogle.bigquery.Table(name="alerts_v10_0", dataset="lsst", projectid=pgb_project)
alerts_tbl.schema

,name,type,mode,description
0,diaSourceId,INTEGER,REQUIRED,Identifier of the triggering DiaSource
1,observation_reason,STRING,NULLABLE,Scheduler reason for the image containing this...
2,target_name,STRING,NULLABLE,Scheduler target for the image containing this...
3,diaSource,RECORD,REQUIRED,
4,diaSource.diaSourceId,INTEGER,REQUIRED,Unique identifier of this DiaSource.
...,...,...,...,...
392,mpc_orbits.fitting_datetime,TIMESTAMP,NULLABLE,Date of the last orbit fit
393,healpix9,INTEGER,REQUIRED,HEALPix order 9 pixel index at the source’s ri...
394,healpix19,INTEGER,REQUIRED,HEALPix order 19 pixel index at the source’s r...
395,healpix29,INTEGER,REQUIRED,HEALPix order 29 pixel index at the source’s r...


<a id="value-added-tables"></a>
## Value-added tables

### SuperNNova

Classification results from SuperNNova ([Möller & de Boissière 2019](https://arxiv.org/pdf/1901.06384)). Explore the schema of the `supernnova` table in the cell below:

In [3]:
supernnova_tbl = pittgoogle.bigquery.Table(name="supernnova", dataset="lsst", projectid=pgb_project)
supernnova_tbl.schema

,name,type,mode,description
0,diaSourceId,INTEGER,REQUIRED,Unique identifier of this DiaSource.
1,diaObjectId,INTEGER,NULLABLE,Unique identifier of this DiaObject.
2,ssObjectId,INTEGER,NULLABLE,Unique identifier of the object.
3,prob_class0,FLOAT,NULLABLE,probability object is a type Ia supernova
4,prob_class1,FLOAT,NULLABLE,probability object is non-Ia
5,predicted_class,INTEGER,NULLABLE,"predicted class; 0 = Ia, 1 = non-Ia"
6,kafkaPublishTimestamp,TIMESTAMP,REQUIRED,Kafka timestamp from originating LSST alert.


### UPSILoN

Classification results from UPSILoN ([Kim & Bailer-Jones 2015](https://arxiv.org/pdf/1512.01611)). Explore the schema of the `upsilon` table in the cell below:

In [4]:
upsilon_tbl = pittgoogle.bigquery.Table(name="upsilon", dataset="lsst", projectid=pgb_project)
upsilon_tbl.schema

,name,type,mode,description
0,diaSourceId,INTEGER,REQUIRED,Unique identifier of this DiaSource.
1,diaObjectId,INTEGER,NULLABLE,Unique identifier of this DiaObject.
2,ssObjectId,INTEGER,NULLABLE,Unique identifier of the object.
3,u_label,STRING,NULLABLE,Predicted class using u band photometry.
4,g_label,STRING,NULLABLE,Predicted class using g band photometry.
5,r_label,STRING,NULLABLE,Predicted class using r band photometry.
6,i_label,STRING,NULLABLE,Predicted class using i band photometry.
7,z_label,STRING,NULLABLE,Predicted class using z band photometry.
8,y_label,STRING,NULLABLE,Predicted class using y band photometry.
9,u_probability,FLOAT,NULLABLE,Class probability for u_label.


### Variability

The following table contains Stetson's J variability index ([Stetson 1996](https://iopscience.iop.org/article/10.1086/133808/pdf)). Explore the schema of the `variability` table in the cell below:

In [5]:
variability_tbl = pittgoogle.bigquery.Table(name="variability", dataset="lsst", projectid=pgb_project)
variability_tbl.schema

,name,type,mode,description
0,diaSourceId,INTEGER,REQUIRED,Unique identifier of this DiaSource.
1,diaObjectId,INTEGER,NULLABLE,Unique identifier of this DiaObject.
2,ssObjectId,INTEGER,NULLABLE,Unique identifier of the object.
3,u_psfFluxStetsonJ,FLOAT,NULLABLE,Stetson J index calculated using the u band DI...
4,g_psfFluxStetsonJ,FLOAT,NULLABLE,Stetson J index calculated using the g band DI...
5,r_psfFluxStetsonJ,FLOAT,NULLABLE,Stetson J index calculated using the r band DI...
6,i_psfFluxStetsonJ,FLOAT,NULLABLE,Stetson J index calculated using the i band DI...
7,z_psfFluxStetsonJ,FLOAT,NULLABLE,Stetson J index calculated using the z band DI...
8,y_psfFluxStetsonJ,FLOAT,NULLABLE,Stetson J index calculated using the y band DI...
9,n_detections_u_band,INTEGER,NULLABLE,Number of detections of this object in the u b...


<a id="sql-query-examples"></a>
## SQL query examples

[SQL queries](https://www.geeksforgeeks.org/sql/what-is-sql/) can be used to access the data stored in the BigQuery tables served by the Pitt-Google Alert Broker. By default the first 1 TiB of query data processed per month is free for every Google Cloud Platform user. See [here](https://mwvgroup.github.io/pittgoogle-client/faq/cost.html) for more information on costs.

In [6]:
# the query below returns the distinct date values in the kafkaPublishTimestamp field of the alerts table
alerts_tbl.query(columns=["DISTINCT(DATE(kafkaPublishTimestamp))"])

,f0_
0,2026-02-17
1,2026-02-27
2,2026-03-06
3,2026-02-26
4,2026-03-07
5,2026-03-04
6,2026-03-09
7,2026-02-13
8,2026-03-10
9,2026-02-20


As we see in the cell above, the `alerts_v10_0` contains alert data published on various dates. It will sometimes be useful to construct queries containing date ranges to optimize query performance. An example is outlined below:

In [7]:
column = ["DISTINCT(diaObject.diaObjectId)"]
where = 'DATE(kafkaPublishTimestamp) BETWEEN "2026-02-28" AND "2026-03-08"'
alerts_tbl.query(columns=column, where=where)

,diaObjectId
0,313963359394857049
1,313761042278645849
2,170072470619947061
3,170028491768594569
4,170028487471530016
...,...
11058,313928195562274884
11059,313862199489069145
11060,313681914488160388
11061,313774266564214786


In [8]:
# define query parameters
column = ["diaSource.psfFlux, diaSource.psfFluxErr, diaSource.midpointMjdTai, diaSource.band"]
where = 'DATE(kafkaPublishTimestamp) BETWEEN "2026-02-28" AND "2026-03-08" AND diaObject.diaObjectId = 313761042278645849'

# execute query
photometry = alerts_tbl.query(columns=column, where=where)
photometry

,psfFlux,psfFluxErr,midpointMjdTai,band
0,-3710.872559,349.266632,61101.030547,i
1,-4521.435059,270.710846,61099.039898,g
2,-3621.901367,437.250824,61105.093204,i
3,-5028.177246,491.984924,61105.103776,g
4,-4822.755859,339.508087,61100.077972,r
5,-4938.512207,217.726883,61107.036174,g


An alternative method of writing and executing SQL queries can be done using the BigQuery client. This is the appropriate choice to make when writing more complicated queries (e.g., JOIN statements).

In [9]:
bqclient = pittgoogle.bigquery.Client() # load the BigQuery client

sql_query = f"""
SELECT
  alerts.diaObject.diaObjectId,
  alerts.diaSource.psfFlux AS psfFlux,
  alerts.diaSource.psfFluxErr AS psfFluxErr,
  alerts.diaSource.band,
  snn.prob_class0
FROM `pitt-alert-broker.lsst.alerts_v10_0` AS alerts
INNER JOIN `pitt-alert-broker.lsst.supernnova` AS snn
  ON alerts.diaObject.diaObjectId = snn.diaObjectId
WHERE
  snn.prob_class0 > 0.9
  AND (DATE(alerts.kafkaPublishTimestamp) BETWEEN "2026-02-28" AND "2026-03-08")
ORDER BY alerts.diaSource.diaObjectId
"""

bqclient.query(query=sql_query)

,diaObjectId,psfFlux,psfFluxErr,band,prob_class0
0,170028485695766556,-12776.604492,474.762329,r,0.900483
1,170028485906530388,14943.814453,703.176758,r,0.985591
2,170028485906530388,17296.494141,432.895782,g,0.985591
3,170028485965774949,-1759.409546,307.636169,g,0.913837
4,170028485980454988,-4008.361572,341.192841,g,0.915667
...,...,...,...,...,...
409,313985344781418501,156644.375000,1048.506348,g,0.965283
410,313985344781418501,138577.046875,1069.300659,g,0.965283
411,313985344781418501,138577.046875,1069.300659,g,0.974148
412,313998539476173288,1556.035156,286.874329,g,0.901182
